# 珠子初始定价分析（2026-07-22）

## tl;dr

测试环境当前启用的 39 个圆珠品种、312 个 8–15mm SKU 已建立按品种与咪数区分的单颗初始价。定价以公开散珠市场锚点为参考，并按普通、精选/包体、稀缺发晶三类尺寸曲线递增；结果用于首版运营价，不替代具体批次的成本和品相评估。

## Context & Methods

- 口径：单颗人民币零售价，保留到 0.5 元；仅覆盖当前启用的圆珠 SKU。
- 市场参考：[义乌购多品种散珠](https://www.yiwugo.com/product/detail/985484652.html)、[白水晶/白阿塞/月光石](https://yiwugo.com/product/detail/980219975.html)、[绿幽灵](https://www.yiwugo.com/product/detail/958877393.html)、[发晶/兔毛/超七等](https://www.yiwugo.com/product/detail/976675970.html)。
- 品质约束：[GIA 海蓝宝购买指南](https://www.gia.edu/aquamarine/buyers-guide) 与 [GIA 紫水晶品质因素](https://www.gia.edu/amethyst-quality-factor?lang=en) 用于确认颜色、净度和处理会显著影响价值。
- 计算：每个品种设置 8mm 锚点，再应用 C/M/R 三条 8–15mm 尺寸曲线，最后四舍五入到 0.5 元。

## Data

In [1]:
from collections import defaultdict
from statistics import median

from scripts.set_initial_bead_prices import (
    REQUIRED_SIZES,
    SERIES_PRICE_PROFILES,
    price_for_series_size,
)

rows = [
    {
        'series': series,
        'curve': curve,
        'anchor_8mm': float(anchor),
        'size_mm': size,
        'price': float(price_for_series_size(series, size)),
    }
    for series, (curve, anchor) in SERIES_PRICE_PROFILES.items()
    for size in REQUIRED_SIZES
]
print({'sku_count': len(rows), 'series_count': len(SERIES_PRICE_PROFILES), 'sizes_mm': list(REQUIRED_SIZES)})

{'sku_count': 312, 'series_count': 39, 'sizes_mm': [8, 9, 10, 11, 12, 13, 14, 15]}


## Results

In [2]:
prices_by_size = defaultdict(list)
for row in rows:
    prices_by_size[row['size_mm']].append(row['price'])

size_summary = [
    {
        'size_mm': size,
        'min_price': min(prices),
        'median_price': median(prices),
        'max_price': max(prices),
    }
    for size, prices in sorted(prices_by_size.items())
]
for item in size_summary:
    print(item)

{'size_mm': 8, 'min_price': 2.0, 'median_price': 7.0, 'max_price': 20.0}
{'size_mm': 9, 'min_price': 2.5, 'median_price': 9.0, 'max_price': 27.0}
{'size_mm': 10, 'min_price': 3.0, 'median_price': 11.5, 'max_price': 35.0}
{'size_mm': 11, 'min_price': 4.0, 'median_price': 14.5, 'max_price': 44.5}
{'size_mm': 12, 'min_price': 4.5, 'median_price': 17.5, 'max_price': 55.0}
{'size_mm': 13, 'min_price': 5.5, 'median_price': 21.0, 'max_price': 67.5}
{'size_mm': 14, 'min_price': 6.0, 'median_price': 24.5, 'max_price': 81.0}
{'size_mm': 15, 'min_price': 7.0, 'median_price': 29.0, 'max_price': 96.5}


In [3]:
focus_series = ['白水晶', '白阿塞', '绿幽灵', '金发晶', '钛晶']
for series in focus_series:
    prices = {size: float(price_for_series_size(series, size)) for size in (8, 10, 12, 15)}
    print(series, prices)

白水晶 {8: 2.5, 10: 4.0, 12: 5.5, 15: 9.0}
白阿塞 {8: 4.0, 10: 6.0, 12: 9.0, 15: 14.0}
绿幽灵 {8: 9.0, 10: 15.0, 12: 22.5, 15: 37.0}
金发晶 {8: 16.5, 10: 29.0, 12: 45.5, 15: 79.5}
钛晶 {8: 20.0, 10: 35.0, 12: 55.0, 15: 96.5}


## Takeaways

1. 所有品种 8–15mm 价格严格递增，避免不同咪数同价。
2. 普通石英类保持亲民，发晶、钛晶等大珠径因成材率和稀缺性采用更陡曲线。
3. 这是可运营的初始价格带；正式销售前仍应结合每批采购成本、净度、颜色、证书和目标毛利二次校准。
4. 当前历史停用 SKU 不在本次范围内，避免覆盖旧数据。